# Structural Causal Bottelneck data generation tutorial
In this tutorial, we will familiarize ourselves with the basics of defining structural causal bottleneck models and generating data from them.

We will use a simple two node example with the structure $X_1 \rightarrow X_2$, where $X_1$ and $X_2$ are vector valued macro variables with internal microstates.

In [1]:
# Imports and global vars

import numpy as np

from cbm import sample_mrf_prec, GaussianLangevinMechanism, MacroCausalVar, SCBM, Intervention

seed = 0
rs = np.random.RandomState(seed=seed)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


First, let's define the root node $X_1$. To do so, we use the `MacroCausalVar` class:

In [2]:
n1 = 4 # number of internal states

# We use the same sparsity structure within each node for simplicity. This is defined by the following binary mask:
M = np.ones((n1, n1))
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0

# Next, we sample a precision matrix that corresponds to a Gaussian Markov Random field
P1 = sample_mrf_prec(dim=n1, M=M, rs=rs)

# Define the internal mechanism of the node X1
mech1 = GaussianLangevinMechanism(mu=np.zeros(n1), E=np.linalg.inv(P1))

X1 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech1, n=n1)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


The second variable $X_2$ is defined similarly, although now we define a *bottleneck function* that induces the distribution of the bottleneck variable $Y_{12}$. Additionally, the mean vector of $X_2$ is a function that takes the values of $Y_{12}$ as arguments:

In [3]:
n2 = 4

# Bottleneck function
f_y12 = lambda x: np.mean(x, axis=1)

P2 = sample_mrf_prec(dim=n2, M=M, rs=rs)

# Need list comprehension to deal with multiple samples as single x input
mu2 = lambda x: [np.sin(x_elem) * np.ones(n2) for x_elem in x]

mech1 = GaussianLangevinMechanism(mu=mu2, E=np.linalg.inv(P2))

X2 = MacroCausalVar(parents=[X1], bottleneck_fcts=[f_y12], mechanism=mech1, n=n2)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


Now, we are ready to define a Structural Causal Bottleneck Model. We do so with the help of the `SCBM` class:

In [4]:
toy_scbm = SCBM(variables=[X1, X2], seed=seed) # variables must be in correct causal order!

With this object, we can now draw a sample from the observational distribution:

In [5]:
sample_obs = toy_scbm.sample(size=100)

Each sample is a list with len = nr. of macro variables, where each element is an array of size = [nr. of samples, nr. of micro variables]:

In [6]:
print(F"Elements in sample list: {len(sample_obs)}")
print(F"Size of x_1: {sample_obs[1].shape}")

Elements in sample list: 2
Size of x_1: (100, 4)


To draw samples from interventional distributions we first define an intervention with the `Intervention` class:

In [7]:
toy_iv = Intervention(macro_targets=1, micro_targets=[2, 3], values=[2, 2])
test_sample_iv = toy_scbm.intervent_sample(iv=toy_iv, size=100)